# Convolutional Neural Networks (CNNs)

## Learning Objectives
1. Implement a 2D convolution from scratch using NumPy and verify output shapes.
2. Build and train a CNN in PyTorch with OOM-safe training loop.
3. Apply transfer learning from a pretrained ResNet-style model to a new task.
4. Visualise intermediate feature maps to understand what each layer detects.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: 2D Convolution from Scratch (NumPy)

In [ ]:
# Implement valid 2D convolution and max pooling from scratch.

def conv2d_scratch(image, kernel, stride=1):
    # 2D convolution (valid padding, single channel).
    # image: (H, W), kernel: (kH, kW)
    # Returns: ((H-kH)//stride + 1, (W-kW)//stride + 1)
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    output = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            patch = image[i*stride : i*stride+kH, j*stride : j*stride+kW]
            output[i, j] = (patch * kernel).sum()
    return output

def maxpool2d_scratch(x, pool=2):
    H, W = x.shape
    oh, ow = H // pool, W // pool
    return x[:oh*pool, :ow*pool].reshape(oh, pool, ow, pool).max(axis=(1, 3))

# Verify with Sobel-X edge-detection kernel
image = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0],
], dtype=float)
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)

out = conv2d_scratch(image, sobel_x)
print(f"Input shape : {image.shape}")
print(f"Kernel shape: {sobel_x.shape}")
print(f"Output shape: {out.shape}  (valid padding: H-kH+1 x W-kW+1)")
print(f"Sobel-X output (positive=right edge, negative=left edge):")
print(out.astype(int))

pooled = maxpool2d_scratch(out, pool=2)
print(f"After 2x2 max pooling: shape={pooled.shape}  values={pooled.flatten().round(1)}")


## Level 2: CNN in PyTorch with OOM-Safe Training

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Synthetic 32x32 grayscale dataset, 10 classes
X_cnn = torch.randn(500, 1, 32, 32)
y_cnn = torch.randint(0, 10, (500,))
tr_ds = TensorDataset(X_cnn[:400].to(device), y_cnn[:400].to(device))
va_ds = TensorDataset(X_cnn[400:].to(device), y_cnn[400:].to(device))
tr_ld = DataLoader(tr_ds, batch_size=32, shuffle=True)
va_ld = DataLoader(va_ds, batch_size=32)

cnn = SimpleCNN(n_classes=10).to(device)
opt_cnn = torch.optim.Adam(cnn.parameters(), lr=1e-3)
crit_cnn = nn.CrossEntropyLoss()

for epoch in range(15):
    cnn.train()
    for xb, yb in tr_ld:
        opt_cnn.zero_grad()
        try:
            loss = crit_cnn(cnn(xb), yb)
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print("OOM -- reduce batch size or model size")
                torch.cuda.empty_cache()
                continue
            raise
        loss.backward()
        opt_cnn.step()

cnn.eval()
with torch.no_grad():
    correct = sum((cnn(xb).argmax(1) == yb).sum().item() for xb, yb in va_ld)
print(f"CNN validation accuracy: {correct / len(va_ds):.4f}")
print(f"Total parameters: {sum(p.numel() for p in cnn.parameters()):,}")


## Real-World Example 1: Transfer Learning from a ResNet-Style Block

In [ ]:
# Simulate transfer learning: freeze pretrained feature extractor,
# replace and train only the classifier head.

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + x)

class TinyResNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1, bias=False),
                                    nn.BatchNorm2d(32), nn.ReLU())
        self.blocks = nn.Sequential(*[ResidualBlock(32) for _ in range(4)])
        self.head   = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, n_classes))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

# Pre-train on task A (5 classes)
model_a = TinyResNet(n_classes=5).to(device)
X_a = torch.randn(300, 1, 32, 32, device=device)
y_a = torch.randint(0, 5, (300,), device=device)
opt_a = torch.optim.Adam(model_a.parameters(), lr=1e-3)
for _ in range(10):
    opt_a.zero_grad()
    nn.CrossEntropyLoss()(model_a(X_a), y_a).backward()
    opt_a.step()

# Transfer: freeze stem + blocks, fine-tune new head for task B (3 classes)
model_b = TinyResNet(n_classes=3).to(device)
model_b.stem.load_state_dict(model_a.stem.state_dict())
model_b.blocks.load_state_dict(model_a.blocks.state_dict())
for p in list(model_b.stem.parameters()) + list(model_b.blocks.parameters()):
    p.requires_grad = False

X_b = torch.randn(200, 1, 32, 32, device=device)
y_b = torch.randint(0, 3, (200,), device=device)
opt_b = torch.optim.Adam(filter(lambda p: p.requires_grad, model_b.parameters()), lr=1e-3)
for _ in range(15):
    opt_b.zero_grad()
    nn.CrossEntropyLoss()(model_b(X_b), y_b).backward()
    opt_b.step()

with torch.no_grad():
    acc_b = (model_b(X_b).argmax(1) == y_b).float().mean().item()
trainable = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model_b.parameters())
print(f"Transfer learning -- Task B accuracy: {acc_b:.4f}")
print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.1f}%)")


## Real-World Example 2: Depthwise Separable Convolution

In [ ]:
# Depthwise separable convolution (MobileNet): depthwise + pointwise.
# Reduces parameters ~8-9x compared to standard conv with similar accuracy.

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        # Depthwise: one filter per input channel (groups=in_ch)
        self.dw    = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.dw_bn = nn.BatchNorm2d(in_ch)
        # Pointwise: 1x1 conv to mix channels
        self.pw    = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.pw_bn = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = F.relu(self.dw_bn(self.dw(x)))
        return F.relu(self.pw_bn(self.pw(x)))

x_dw = torch.randn(4, 32, 16, 16, device=device)

std_conv = nn.Sequential(
    nn.Conv2d(32, 64, 3, padding=1, bias=False),
    nn.BatchNorm2d(64), nn.ReLU()
).to(device)
dws_conv = DepthwiseSeparableConv(32, 64).to(device)

p_std = sum(p.numel() for p in std_conv.parameters())
p_dws = sum(p.numel() for p in dws_conv.parameters())
with torch.no_grad():
    out_std = std_conv(x_dw)
    out_dws = dws_conv(x_dw)
print(f"Standard conv       -- params: {p_std:,}  out: {tuple(out_std.shape)}")
print(f"Depthwise-separable -- params: {p_dws:,}  out: {tuple(out_dws.shape)}")
print(f"Parameter reduction: {p_std/p_dws:.2f}x  (same output shape, fewer weights)")


## Real-World Example 3: Feature Map Visualization

In [ ]:
# Hook into conv layers to extract and inspect feature maps.
# Early layers detect edges; later layers detect semantic patterns.

feature_maps = {}

def register_hooks(model):
    handles = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            def make_hook(n):
                def hook(_, __, output):
                    feature_maps[n] = output.detach().cpu()
                return hook
            handles.append(module.register_forward_hook(make_hook(name)))
    return handles

vis_cnn = SimpleCNN(n_classes=5).to(device)
x_vis = torch.randn(1, 1, 32, 32, device=device)

handles = register_hooks(vis_cnn)
with torch.no_grad():
    _ = vis_cnn(x_vis)
for h in handles:
    h.remove()

print("Feature maps extracted from conv layers:")
for name, fmap in feature_maps.items():
    B, C, H, W = fmap.shape
    act_mean  = fmap.abs().mean().item()
    sparsity  = (fmap == 0).float().mean().item()
    print(f"  {name:40s}  C={C}  {H}x{W}  mean_act={act_mean:.4f}  sparsity={sparsity:.2%}")

mags = [feature_maps[n].abs().mean().item() for n in feature_maps]
print(f"Activation magnitude trend: {[round(m, 3) for m in mags]}")
print("Early layers: higher raw activation (raw edges).")
print("Later layers: sparser but more semantically meaningful features.")
